In [1]:
"""
04_world_cup_simulation.py — 2026 FIFA World Cup Monte Carlo Simulation.

Builds proper feature vectors for 2026 matches by reusing the same
data lookups as 02_build_features.py (Elo, FIFA rank, squad values,
form, H2H, WC history), then runs 10,000 Monte Carlo iterations.
"""

'\n04_world_cup_simulation.py — 2026 FIFA World Cup Monte Carlo Simulation.\n\nBuilds proper feature vectors for 2026 matches by reusing the same\ndata lookups as 02_build_features.py (Elo, FIFA rank, squad values,\nform, H2H, WC history), then runs 10,000 Monte Carlo iterations.\n'

In [2]:
import sys
from pathlib import Path

In [3]:
BASE = (Path.cwd() if (Path.cwd() / 'data' / 'processed').exists() else Path.cwd().parent)
sys.path.insert(0, str(BASE))

In [4]:
import numpy as np
import pandas as pd
import joblib
from collections import defaultdict, deque
import bisect

In [5]:
from src.simulation import (simulate_group_stage, get_advancing_teams,
                             slot_third_place_teams, simulate_knockout_match)

In [6]:
# ── Paths ──────────────────────────────────────────────────────────────
WC_DIR = BASE / "data" / "raw" / "world_cup_2026"
RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"
ARTIFACTS = BASE / "outputs" / "model_artifacts"
SIM_DIR = BASE / "outputs" / "simulation_results"
SIM_DIR.mkdir(parents=True, exist_ok=True)

In [7]:
print("=" * 60)
print("2026 FIFA World Cup Monte Carlo Simulation")
print("=" * 60)

2026 FIFA World Cup Monte Carlo Simulation


In [8]:
# ── Load tournament structure ──────────────────────────────────────────
fixtures = pd.read_csv(WC_DIR / "fixtures.csv")
groups = pd.read_csv(WC_DIR / "groups.csv")
bracket = pd.read_csv(WC_DIR / "knockout_bracket.csv")
all_teams = groups["team"].tolist()

In [9]:
print(f"Groups: {len(groups)} teams, {len(fixtures)} group matches, {len(bracket)} knockout matches")

Groups: 48 teams, 72 group matches, 32 knockout matches


In [10]:
# ── Load model ─────────────────────────────────────────────────────────
model = joblib.load(ARTIFACTS / "XGBoost_tuned_balanced.joblib")
print(f"Model loaded: XGBoost_tuned_balanced")

Model loaded: XGBoost_tuned_balanced


In [11]:
# ══════════════════════════════════════════════════════════════════════
# LOAD RAW DATA & BUILD TEAM STATE (same approach as 02_build_features.py)
# ══════════════════════════════════════════════════════════════════════
print("\nLoading raw data sources...")


Loading raw data sources...


In [12]:
# Import name mappings from feature builder
sys.path.insert(0, str(BASE / "data" / "scripts"))
from importlib import import_module
fb_spec = import_module("02_build_features")

In [13]:
# ── 1. Match results + Elo ─────────────────────────────────────────────
results = fb_spec.load_results()
print(f"  Match results: {len(results)} matches")

  Match results: 49215 matches


In [14]:
print("  Computing Elo ratings from scratch...")
elo_hist = fb_spec.compute_elo_ratings(results)

  Computing Elo ratings from scratch...


  Computing Elo ratings:   0%|          | 0/49215 [00:00<?, ?it/s]

  Computing Elo ratings:   7%|▋         | 3563/49215 [00:00<00:01, 35627.15it/s]

  Computing Elo ratings:  20%|█▉        | 9824/49215 [00:00<00:00, 51498.27it/s]

  Computing Elo ratings:  33%|███▎      | 16053/49215 [00:00<00:00, 56424.68it/s]

...

  Computing Elo ratings:  58%|█████▊    | 28603/49215 [00:00<00:00, 60308.48it/s]

  Computing Elo ratings:  71%|███████   | 34891/49215 [00:00<00:00, 61182.19it/s]

  Computing Elo ratings:  83%|████████▎ | 41062/49215 [00:00<00:00, 61348.69it/s]

  Computing Elo ratings:  96%|█████████▌| 47335/49215 [00:00<00:00, 61787.85it/s]

  Computing Elo ratings: 100%|██████████| 49215/49215 [00:00<00:00, 59135.51it/s]

In [15]:
# Current Elo for each team (last recorded value)
current_elo = {}
for team in all_teams:
    if team in elo_hist and len(elo_hist[team]) > 0:
        current_elo[team] = elo_hist[team][-1][1]
    else:
        current_elo[team] = 1500.0  # default
        print(f"    WARNING: no Elo history for {team}, using 1500")

In [16]:
# Print Elo rankings for 48 teams
elo_ranked = sorted(current_elo.items(), key=lambda x: -x[1])
print("\n  Current Elo ratings (48 teams):")
for i, (team, elo) in enumerate(elo_ranked):
    print(f"    {i+1:2d}. {team:<28s} {elo:.0f}")


  Current Elo ratings (48 teams):
     1. Spain                        2198
     2. Argentina                    2152
     3. France                       2127
     4. England                      2085
     5. Colombia                     2057
     6. Brazil                       2034
     7. Ecuador                      2001
     8. Netherlands                  1998
     9. Portugal                     1988
    10. Germany                      1985
    11. Croatia                      1976
    12. Uruguay                      1969
    13. Morocco                      1959
    14. Japan                        1956
    15. Switzerland                  1952
    16. Norway                       1942
    17. Belgium                      1933
    18. Senegal                      1928
    19. Mexico                       1920
    20. Paraguay                     1918
    21. South Korea                  1888
    22. Australia                    1876
    23. Canada                       1875

In [17]:
# Elo momentum (last 5 matches)
current_momentum = {}
for team in all_teams:
    mom = fb_spec.get_elo_momentum(team, pd.Timestamp("2026-06-10"), elo_hist, n=5)
    current_momentum[team] = mom if mom is not None else 0.0

In [18]:
# ── 2. FIFA Rankings ───────────────────────────────────────────────────
print("\n  Loading FIFA rankings...")
fifa_dates, fifa_by_date = fb_spec.load_fifa_rankings()
ref_date = pd.Timestamp("2026-06-10")


  Loading FIFA rankings...


In [19]:
current_rank = {}
current_points = {}
for team in all_teams:
    r, p, _ = fb_spec.get_fifa(team, ref_date, fifa_dates, fifa_by_date)
    current_rank[team] = r if r is not None else 100
    current_points[team] = p if p is not None else 1000.0

In [20]:
# ── 3. Squad values ───────────────────────────────────────────────────
print("  Loading Transfermarkt squad values...")
players, vals = fb_spec.load_players_and_valuations()
squad_agg, player_snap, top5_snap, quarters = fb_spec.precompute_squad_snapshots(
    players, vals, pd.Timestamp("2004-01-01")
)

  Loading Transfermarkt squad values...
  Joining players with valuations...


  Quarterly squad values:   0%|          | 0/92 [00:00<?, ?it/s]

  Quarterly squad values:   9%|▊         | 8/92 [00:00<00:01, 71.75it/s]

  Quarterly squad values:  17%|█▋        | 16/92 [00:00<00:03, 23.21it/s]

  Quarterly squad values:  22%|██▏       | 20/92 [00:00<00:03, 20.81it/s]

...

  Quarterly squad values:  96%|█████████▌| 88/92 [00:07<00:00,  8.85it/s]

  Quarterly squad values:  97%|█████████▋| 89/92 [00:07<00:00,  9.03it/s]

  Quarterly squad values:  98%|█████████▊| 90/92 [00:07<00:00,  9.23it/s]

  Quarterly squad values: 100%|██████████| 92/92 [00:08<00:00,  7.45it/s]

  Quarterly squad values: 100%|██████████| 92/92 [00:08<00:00, 11.34it/s]

In [21]:
current_squad = {}
for team in all_teams:
    sq = fb_spec.get_squad(team, ref_date, squad_agg, quarters)
    current_squad[team] = sq  # can be None

In [22]:
# ── 4. Form (rolling 10 matches) ──────────────────────────────────────
print("  Computing form from match history...")
IMPORTANCE = fb_spec.IMPORTANCE
form_tracker = defaultdict(lambda: deque(maxlen=10))
last_match_date = {}

  Computing form from match history...


In [23]:
for _, row in results.iterrows():
    home, away = row["home_team"], row["away_team"]
    hs, aws = int(row["home_score"]), int(row["away_score"])
    date = row["date"]
    imp = IMPORTANCE.get(row["tournament"], 0.3)

    h_result = "win" if hs > aws else ("draw" if hs == aws else "loss")
    a_result = "win" if aws > hs else ("draw" if hs == aws else "loss")

    form_tracker[home].append({"r": h_result, "gf": hs, "ga": aws, "imp": imp})
    form_tracker[away].append({"r": a_result, "gf": aws, "ga": hs, "imp": imp})
    last_match_date[home] = date
    last_match_date[away] = date

In [24]:
def compute_form_stats(team):
    """Compute win rate, weighted form, and goal diff from rolling form."""
    team_form = form_tracker.get(team)
    if not team_form or len(team_form) == 0:
        return 0.5, 0.25, 0.0, 30.0  # defaults

    wins = sum(1 for m in team_form if m["r"] == "win")
    wr = wins / len(team_form)

    wts = []
    for i, m in enumerate(reversed(list(team_form))):
        decay = 0.9 ** i
        score = 1.0 if m["r"] == "win" else (0.5 if m["r"] == "draw" else 0.0)
        wts.append(decay * m["imp"] * score)
    wf = np.mean(wts) if wts else 0.25

    gds = [m["gf"] - m["ga"] for m in team_form]
    gd = np.mean(gds)

    days_rest = (ref_date - last_match_date.get(team, ref_date - pd.Timedelta(days=30))).days

    return wr, wf, gd, days_rest

In [25]:
# ── 5. H2H ─────────────────────────────────────────────────────────────
print("  Building H2H history...")
h2h_db = defaultdict(list)
for _, row in results.iterrows():
    home, away = row["home_team"], row["away_team"]
    hs, aws = int(row["home_score"]), int(row["away_score"])
    pair = tuple(sorted([home, away]))
    winner = home if hs > aws else (away if aws > hs else None)
    h2h_db[pair].append({"date": row["date"], "winner": winner, "home": home})

  Building H2H history...


In [26]:
# ── 6. World Cup history ──────────────────────────────────────────────
print("  Loading World Cup history...")
wc_data = fb_spec.load_wc_history()

  Loading World Cup history...


In [27]:
# ── 7. Confederation lookup ───────────────────────────────────────────
team_conf = {}
for dd in fifa_by_date.values():
    for t, d in dd.items():
        if d.get("conf"):
            team_conf[t] = d["conf"]

══════════════════════════════════════════════════════════════════════
BUILD FEATURE VECTORS FOR 2026 MATCHES
══════════════════════════════════════════════════════════════════════

In [28]:
def log_transform(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return np.nan
    return np.sign(x) * np.log1p(abs(x))

In [29]:
def build_match_features(home, away):
    """Build a proper feature vector for a match, same logic as 02_build_features.py."""
    f = {}

    # CONTEXT
    f["match_importance"] = 1.0  # World Cup
    f["is_neutral"] = 1          # All WC matches are neutral venue

    h_conf = team_conf.get(home, "")
    a_conf = team_conf.get(away, "")
    f["same_confederation"] = int(h_conf != "" and h_conf == a_conf)
    f["home_continent_advantage"] = 0  # neutral venue

    # ELO
    h_elo = current_elo.get(home, 1500)
    a_elo = current_elo.get(away, 1500)
    f["elo_diff"] = h_elo - a_elo

    h_mom = current_momentum.get(home, 0)
    a_mom = current_momentum.get(away, 0)
    f["elo_momentum_diff"] = h_mom - a_mom

    # FIFA RANKINGS
    h_rank = current_rank.get(home, 100)
    a_rank = current_rank.get(away, 100)
    f["rank_diff"] = h_rank - a_rank

    h_pts = current_points.get(home, 1000)
    a_pts = current_points.get(away, 1000)
    f["points_diff"] = h_pts - a_pts

    # SQUAD VALUES
    h_sq = current_squad.get(home)
    a_sq = current_squad.get(away)
    if h_sq and a_sq:
        for key in ["total", "top11", "attack", "mid", "def"]:
            f[f"squad_value_{key}_delta"] = log_transform(h_sq[key] - a_sq[key])
        f["star_player_value_delta"] = log_transform(h_sq["star"] - a_sq["star"])
        f["squad_depth_delta"] = log_transform(h_sq["depth"] - a_sq["depth"])
    else:
        for col in ["squad_value_total_delta", "squad_value_top11_delta",
                     "squad_value_attack_delta", "squad_value_mid_delta",
                     "squad_value_def_delta", "star_player_value_delta",
                     "squad_depth_delta"]:
            f[col] = np.nan

    # FORM
    h_wr, h_wf, h_gd, h_rest = compute_form_stats(home)
    a_wr, a_wf, a_gd, a_rest = compute_form_stats(away)
    f["home_days_rest"] = h_rest
    f["away_days_rest"] = a_rest
    f["form_win_rate_diff"] = h_wr - a_wr
    f["form_weighted_diff"] = h_wf - a_wf
    f["goal_diff_delta"] = h_gd - a_gd

    # H2H
    pair = tuple(sorted([home, away]))
    hist = h2h_db.get(pair, [])
    if hist:
        h_wins = sum(1 for h in hist if h["winner"] == home)
        draws = sum(1 for h in hist if h["winner"] is None)
        f["h2h_home_win_rate"] = h_wins / len(hist)
        f["h2h_draw_rate"] = draws / len(hist)
        f["h2h_matches_played"] = len(hist)
    else:
        f["h2h_home_win_rate"] = np.nan
        f["h2h_draw_rate"] = np.nan
        f["h2h_matches_played"] = 0

    # TOURNAMENT WIN RATE (use WC-specific)
    f["tournament_wr_delta"] = 0.0  # no prior WC 2026 matches to base this on

    # WC HISTORY
    h_wc = wc_data.get(home, {})
    a_wc = wc_data.get(away, {})
    for key, default in [("appearances", 0), ("knockout_rate", 0),
                          ("best_finish", 1), ("goals_per_game", 0)]:
        f[f"wc_{key}_diff"] = h_wc.get(key, default) - a_wc.get(key, default)

    # INJURIES (set to 0 — no live scrape yet)
    f["injury_count_delta"] = 0
    f["injury_burden_delta"] = 0.0
    f["star_injury_flag"] = 0

    return f

In [30]:
# ── Get the expected column order from the training data ───────────────
feature_cols = pd.read_csv(PROCESSED / "splits" / "X_train.csv", nrows=0).columns.tolist()

In [31]:
print(f"\nBuilding feature vectors for {len(fixtures)} group stage matches...")


Building feature vectors for 72 group stage matches...


In [32]:
match_probas = {}
match_details = []

In [33]:
for _, match in fixtures.iterrows():
    mn = match["match_number"]
    home = match["home_team"]
    away = match["away_team"]

    f = build_match_features(home, away)
    X = pd.DataFrame([f]).reindex(columns=feature_cols)
    proba = model.predict_proba(X)[0]

    match_probas[mn] = (proba[0], proba[1], proba[2])
    match_details.append({
        "match_number": mn, "home_team": home, "away_team": away,
        "group": match["group"],
        "p_home_win": proba[0], "p_draw": proba[1], "p_away_win": proba[2],
        "predicted": ["Home Win", "Draw", "Away Win"][np.argmax(proba)]
    })

In [34]:
pred_df = pd.DataFrame(match_details)
pred_df.to_csv(SIM_DIR / "group_match_predictions.csv", index=False)

In [35]:
print("\nGroup stage predictions:")
for _, row in pred_df.iterrows():
    print(f"  M{int(row['match_number']):02d} {row['group']} {row['home_team']:<28s} vs {row['away_team']:<28s} "
          f"[HW:{row['p_home_win']:.2f} D:{row['p_draw']:.2f} AW:{row['p_away_win']:.2f}] -> {row['predicted']}")


Group stage predictions:
  M01 A Mexico                       vs South Africa                 [HW:0.92 D:0.03 AW:0.05] -> Home Win
  M02 A South Korea                  vs Czechia                      [HW:0.78 D:0.19 AW:0.04] -> Home Win
  M03 B Canada                       vs Bosnia and Herzegovina       [HW:0.86 D:0.13 AW:0.01] -> Home Win
  M04 D United States                vs Paraguay                     [HW:0.42 D:0.27 AW:0.31] -> Home Win
  M05 B Qatar                        vs Switzerland                  [HW:0.04 D:0.04 AW:0.92] -> Away Win
  M06 C Brazil                       vs Morocco                      [HW:0.82 D:0.04 AW:0.14] -> Home Win
  M07 C Haiti                        vs Scotland                     [HW:0.12 D:0.08 AW:0.79] -> Away Win
  M08 D Australia                    vs Turkiye                      [HW:0.98 D:0.01 AW:0.00] -> Home Win
  M09 E Germany                      vs Curacao                      [HW:0.98 D:0.02 AW:0.01] -> Home Win
  M10 F Netherlands 

In [36]:
# ── Feature builder for knockout matches ───────────────────────────────
def build_knockout_features(team_a, team_b):
    """Build feature vector for a knockout match."""
    f = build_match_features(team_a, team_b)
    return pd.DataFrame([f]).reindex(columns=feature_cols)

In [37]:
# ══════════════════════════════════════════════════════════════════════
# MONTE CARLO SIMULATION
# ══════════════════════════════════════════════════════════════════════
N_ITERATIONS = 10000
print(f"\nRunning {N_ITERATIONS} Monte Carlo iterations...")


Running 10000 Monte Carlo iterations...


In [38]:
rng = np.random.default_rng(42)

In [39]:
# Use cached simulation results if available
CACHE_FILES = [
    SIM_DIR / "win_probabilities.csv",
    SIM_DIR / "advancement_probabilities.csv",
    SIM_DIR / "group_standings_distribution.csv",
]
USE_CACHE = all(f.exists() for f in CACHE_FILES)
if USE_CACHE:
    print("Loading cached simulation results from outputs/simulation_results/")
    win_probs = pd.read_csv(SIM_DIR / "win_probabilities.csv")
    adv_df = pd.read_csv(SIM_DIR / "advancement_probabilities.csv")
    gs_df = pd.read_csv(SIM_DIR / "group_standings_distribution.csv")
    finals_path = SIM_DIR / "final_matchups.csv"
    final_matchups_df = pd.read_csv(finals_path) if finals_path.exists() else None
else:
    print(f"Running fresh simulation ({N_ITERATIONS} iterations)...")

Loading cached simulation results from outputs/simulation_results/


In [40]:
tournament_wins = defaultdict(int)
round_advancement = defaultdict(lambda: defaultdict(int))
group_standings_dist = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
final_matchups = defaultdict(int)

In [41]:
for team in all_teams:
    for rnd in ["Group", "R32", "R16", "QF", "SF", "Final", "Winner"]:
        round_advancement[team][rnd] = 0

In [42]:
if not USE_CACHE:
    for iteration in range(N_ITERATIONS):
        if iteration % 2000 == 0:
            print(f"  Iteration {iteration}/{N_ITERATIONS}...")
    
        # 1. Group stage
        standings = simulate_group_stage(fixtures, match_probas, groups, rng)
    
        for group, teams in standings.items():
            for pos, t in enumerate(teams):
                group_standings_dist[group][t["team"]][pos + 1] += 1
    
        # 2. Advancing teams
        advancing, third_place_groups = get_advancing_teams(standings)
        for code, team in advancing.items():
            round_advancement[team]["Group"] += 1
    
        # 3. Slot 3rd place
        slot_assignments = slot_third_place_teams(advancing, third_place_groups, bracket)
    
        # 4. Play knockout rounds
        match_winners = {}
    
        for rnd in ["R32", "R16", "QF", "SF", "F"]:
            rnd_label = "Final" if rnd == "F" else rnd
            rnd_matches = bracket[bracket["round"] == rnd]
    
            for _, match in rnd_matches.iterrows():
                mn = match["match_number"]
                home_ref = str(match["home_source"])
                away_ref = str(match["away_source"])
    
                if home_ref.startswith("W") or home_ref.startswith("L"):
                    home_team = match_winners.get(home_ref)
                else:
                    home_team = advancing.get(home_ref)
    
                if "/" in away_ref:
                    code = slot_assignments.get(mn)
                    away_team = advancing.get(code) if code else None
                elif away_ref.startswith("W") or away_ref.startswith("L"):
                    away_team = match_winners.get(away_ref)
                else:
                    away_team = advancing.get(away_ref)
    
                if home_team and away_team:
                    winner = simulate_knockout_match(
                        home_team, away_team, model, build_knockout_features, rng
                    )
                    match_winners[f"W{mn}"] = winner
                    match_winners[f"L{mn}"] = away_team if winner == home_team else home_team
                    round_advancement[winner][rnd_label] += 1
    
                    if rnd == "F":
                        tournament_wins[winner] += 1
                        round_advancement[winner]["Winner"] += 1
                        matchup = tuple(sorted([home_team, away_team]))
                        final_matchups[matchup] += 1

In [43]:
# ══════════════════════════════════════════════════════════════════════
# RESULTS
# ══════════════════════════════════════════════════════════════════════
print("\nProcessing results...")


Processing results...


In [44]:
if not USE_CACHE:
    win_probs = pd.DataFrame([
        {"team": team, "win_probability": count / N_ITERATIONS * 100}
        for team, count in sorted(tournament_wins.items(), key=lambda x: -x[1])
    ])
    # Add teams with 0 wins
    for team in all_teams:
        if team not in win_probs["team"].values:
            win_probs = pd.concat([win_probs, pd.DataFrame([{"team": team, "win_probability": 0.0}])], ignore_index=True)

In [45]:
print("\nTournament Win Probabilities (top 20):")
for _, row in win_probs.head(20).iterrows():
    bar = "#" * int(row["win_probability"] * 2)
    print(f"  {row['team']:<28s} {row['win_probability']:5.1f}% {bar}")


Tournament Win Probabilities (top 20):
  Spain                         16.5% #################################
  Argentina                     11.5% #######################
  Brazil                        11.3% ######################
  France                        10.3% ####################
  England                        8.0% ###############
  Netherlands                    5.3% ##########
  Colombia                       4.6% #########
  Germany                        4.1% ########
  Portugal                       3.9% #######
  Belgium                        3.7% #######
  Norway                         3.0% ######
  Croatia                        2.4% ####
  Japan                          2.2% ####
  Uruguay                        2.0% ###
  Switzerland                    2.0% ###
  Mexico                         1.1% ##
  Ecuador                        1.1% ##
  United States                  1.1% ##
  Morocco                        0.8% #
  Senegal                        0.7% 

In [46]:
if not USE_CACHE:
    adv_data = []
    for team in all_teams:
        row = {"team": team}
        for rnd in ["Group", "R32", "R16", "QF", "SF", "Final", "Winner"]:
            row[rnd] = round_advancement[team].get(rnd, 0) / N_ITERATIONS * 100
        adv_data.append(row)

In [47]:
if not USE_CACHE:
    adv_df = pd.DataFrame(adv_data).sort_values("Winner", ascending=False)
    adv_df.to_csv(SIM_DIR / "advancement_probabilities.csv", index=False)

In [48]:
print("\nAdvancement Probabilities (top 15):")
print(adv_df.head(15)[["team", "Group", "R32", "R16", "QF", "SF", "Final", "Winner"]].to_string(
    index=False, float_format="{:.1f}".format))


Advancement Probabilities (top 15):
       team  Group  R32  R16   QF   SF  Final  Winner
      Spain   99.2 69.6 53.9 38.7 25.6   16.5    16.5
  Argentina   97.5 62.5 43.3 29.9 19.3   11.5    11.5
     Brazil   97.8 66.0 46.0 32.1 19.2   11.3    11.3
     France   95.5 68.0 46.1 29.3 18.7   10.3    10.3
    England   95.6 67.2 47.4 27.2 15.4    8.0     8.0
Netherlands   92.8 54.8 37.5 20.7 10.8    5.3     5.3
   Colombia   94.8 63.0 35.2 19.6  9.4    4.6     4.6
    Germany   96.0 63.8 31.9 18.1  9.1    4.1     4.1
   Portugal   90.9 57.2 31.8 15.9  8.2    3.9     3.9
    Belgium   95.9 62.4 36.9 18.9  8.0    3.7     3.7
     Norway   86.7 53.3 27.2 14.2  6.8    3.0     3.0
    Croatia   90.4 50.5 29.5 12.6  5.7    2.4     2.4
      Japan   85.7 43.5 25.6 11.6  4.9    2.2     2.2
    Uruguay   90.0 42.1 22.6 11.3  4.9    2.0     2.0
Switzerland   91.5 58.0 26.1 10.7  4.8    2.0     2.0


In [49]:
if not USE_CACHE:
    gs_data = []
    for group in sorted(group_standings_dist.keys()):
        for team, positions in group_standings_dist[group].items():
            row = {"group": group, "team": team}
            for pos in [1, 2, 3, 4]:
                row[f"pos_{pos}"] = positions.get(pos, 0) / N_ITERATIONS * 100
            gs_data.append(row)

In [50]:
if not USE_CACHE:
    gs_df = pd.DataFrame(gs_data).sort_values(["group", "pos_1"], ascending=[True, False])
    gs_df.to_csv(SIM_DIR / "group_standings_distribution.csv", index=False)

In [51]:
print("\nGroup Standings (% chance of each position):")
for group in sorted(gs_df["group"].unique()):
    print(f"\n  Group {group}:")
    grp = gs_df[gs_df["group"] == group].sort_values("pos_1", ascending=False)
    for _, row in grp.iterrows():
        print(f"    {row['team']:<28s} 1st:{row['pos_1']:5.1f}%  2nd:{row['pos_2']:5.1f}%  "
              f"3rd:{row['pos_3']:5.1f}%  4th:{row['pos_4']:5.1f}%")


Group Standings (% chance of each position):

  Group A:
    Mexico                       1st: 45.8%  2nd: 29.6%  3rd: 17.7%  4th:  6.9%
    South Korea                  1st: 30.6%  2nd: 33.0%  3rd: 26.0%  4th: 10.4%
    Czechia                      1st: 19.6%  2nd: 27.2%  3rd: 32.3%  4th: 20.9%
    South Africa                 1st:  4.0%  2nd: 10.2%  3rd: 24.1%  4th: 61.7%

  Group B:
    Switzerland                  1st: 48.0%  2nd: 32.4%  3rd: 14.2%  4th:  5.4%
    Canada                       1st: 40.4%  2nd: 37.1%  3rd: 16.4%  4th:  6.1%
    Bosnia and Herzegovina       1st:  7.2%  2nd: 17.4%  3rd: 36.3%  4th: 39.1%
    Qatar                        1st:  4.4%  2nd: 13.1%  3rd: 33.1%  4th: 49.4%

  Group C:
    Brazil                       1st: 67.5%  2nd: 23.8%  3rd:  7.3%  4th:  1.4%
    Morocco                      1st: 22.7%  2nd: 43.2%  3rd: 25.6%  4th:  8.5%
    Scotland                     1st:  8.5%  2nd: 27.0%  3rd: 46.4%  4th: 18.2%
    Haiti                        1st: 

In [52]:
if not USE_CACHE:
    final_matchups_df = pd.DataFrame([
        {"team_a": m[0], "team_b": m[1], "count": c}
        for m, c in sorted(final_matchups.items(), key=lambda x: -x[1])
    ])
    final_matchups_df.to_csv(SIM_DIR / "final_matchups.csv", index=False)

if final_matchups_df is not None and len(final_matchups_df) > 0:
    print()
    print("Most Likely Finals:")
    for _, row in final_matchups_df.head(10).iterrows():
        pct = row["count"] / N_ITERATIONS * 100
        print(f"  {row['team_a']} vs {row['team_b']}: {pct:.2f}%")

In [53]:
if not USE_CACHE:
    win_probs.to_csv(SIM_DIR / "win_probabilities.csv", index=False)

In [54]:
print("\n" + "=" * 60)
print("SIMULATION COMPLETE")
print(f"All results saved to {SIM_DIR}")
print("=" * 60)


SIMULATION COMPLETE
All results saved to /Users/rzein/Desktop/Projects/KickCaster/outputs/simulation_results
